# Hi-EF reliability mechanism diagnostic

This CPU-only notebook runs the frozen v0.7 cross-fitted temperature, residual-shrinkage, and oracle diagnostics. Attach the saved canonical residual matrix output, enable Internet, then choose **Save Version → Save & Run All**. It does not train or evaluate test.

In [ ]:
from pathlib import Path
import json
import os
import subprocess

REPO = Path('/kaggle/working/hi-ef-materials')
OUTPUT = Path('/kaggle/working/canonical_reliability_diagnostic')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', 'experiments', '--single-branch', 'https://github.com/ptrnghieu/hi-ef-materials.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'experiments'], check=True)

candidates = []
for path in Path('/kaggle/input').rglob('canonical_residual_matrix_summary.json'):
    try:
        payload = json.loads(path.read_text())
    except Exception:
        continue
    if payload.get('protocol') == 'canonical-contextual-affective-residual-validation-matrix-v1':
        candidates.append(path)
assert len(candidates) == 1, f'Expected exactly one canonical matrix output, found: {candidates}'
MATRIX = candidates[0].parent
environment = {**os.environ, 'PYTHONPATH': str(REPO / 'experiments')}
print('Commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())
print('Matrix:', MATRIX)


In [ ]:
subprocess.run(['python', '-m', 'unittest', str(REPO / 'experiments/test_canonical_reliability_diagnostics.py')], cwd=REPO, env=environment, check=True)
subprocess.run([
    'python', str(REPO / 'experiments/run_canonical_reliability_diagnostics.py'),
    '--matrix-dir', str(MATRIX),
    '--output-dir', str(OUTPUT),
    '--bootstrap-replicates', '5000',
], cwd=REPO, env=environment, check=True)


In [ ]:
import pandas as pd
summary_path = OUTPUT / 'canonical_reliability_diagnostic_summary.json'
summary = json.loads(summary_path.read_text())
assert summary['diagnostic_only'] is True
assert summary['test_evaluated'] is False
assert summary['partitions_touched'] == ['validation']
display(pd.DataFrame(summary['metric_aggregates']))
print(json.dumps(summary['decision_indicators'], indent=2))
print(json.dumps(summary['hierarchical_bootstrap'], indent=2))
print('Download:', summary_path)
for filename in summary['output_files']:
    print('Download:', OUTPUT / filename)
